# Klasifikasi Bahasa Isyarat Indonesia (BISINDO)
## Perbandingan Algoritma K-Nearest Neighbors, Naive Bayes, dan Support Vector Machine

- **Dataset**: BISINDO Train — 9.173 sampel, 26 kelas alfabet (A–Z), 127 fitur MediaPipe Hand Landmarks
- **Algoritma**: K-Nearest Neighbors (KNN), Naive Bayes (NB), dan Support Vector Machine (SVM)
- **Metrik Evaluasi**: Accuracy, Precision, Recall, F1-Score, dan Stratified 5-Fold Cross-Validation

In [1]:
# Instalasi pustaka (jalankan jika belum terpasang di environment)
# !pip install numpy pandas matplotlib seaborn scikit-learn

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os, time, json

# Scikit-Learn — Model Klasifikasi
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, ComplementNB, BernoulliNB
from sklearn.svm import SVC

# Scikit-Learn — Evaluasi & Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import (
    LabelEncoder, StandardScaler, MinMaxScaler, Binarizer
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("[INFO] Pustaka berhasil dimuat.")
print(f"       NumPy: {np.__version__} | Pandas: {pd.__version__}")

[INFO] Pustaka berhasil dimuat.
       NumPy: 2.4.6 | Pandas: 3.0.3


## 1. Pemuatan dan Eksplorasi Dataset

In [3]:
# Konfigurasi Direktori dan Lokasi Data
BASE_DIR   = r"E:\Dokumen\Kuliah\Semester 5\Pembelajaran Mesin\Klasifikasi_Kelompok_1"
DATA_PATH  = os.path.join(BASE_DIR, "dataset_bisindo_landmarks.npz")
OUTPUT_DIR = os.path.join(BASE_DIR, "output_complete")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Lokasi Dataset : {DATA_PATH}")
print(f"Folder Output  : {OUTPUT_DIR}")
print(f"Status File    : {'Tersedia' if os.path.exists(DATA_PATH) else 'Tidak Ditemukan'}")

Lokasi Dataset : E:\Dokumen\Kuliah\Semester 5\Pembelajaran Mesin\Klasifikasi_Kelompok_1\dataset_bisindo_landmarks.npz
Folder Output  : E:\Dokumen\Kuliah\Semester 5\Pembelajaran Mesin\Klasifikasi_Kelompok_1\output_complete
Status File    : Tersedia


In [4]:
data   = np.load(DATA_PATH, allow_pickle=True)
X      = np.nan_to_num(data['X'].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
labels = data['labels']

le           = LabelEncoder()
y            = le.fit_transform(labels)
class_names  = le.classes_

print(f"{'Dimensi Fitur (X)':<25}: {X.shape}")
print(f"{'Jumlah Total Sampel':<25}: {len(y)}")
print(f"{'Jumlah Kelas':<25}: {len(class_names)}")
print(f"{'Daftar Kelas':<25}: {list(class_names)}")
print()

# Distribusi Sampel per Kelas
unique, counts = np.unique(y, return_counts=True)
df_dist = pd.DataFrame({'Kelas': class_names, 'Jumlah': counts})
print("Distribusi Sampel per Kelas:")
print(df_dist.to_string(index=False))

Dimensi Fitur (X)        : (7893, 127)
Jumlah Total Sampel      : 7893
Jumlah Kelas             : 26
Daftar Kelas             : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

Distribusi Sampel per Kelas:
Kelas  Jumlah
    A     223
    B     306
    C     292
    D     325
    E     311
    F     284
    G     291
    H     282
    I     353
    J     340
    K     260
    L     323
    M     355
    N     350
    O     306
    P     354
    Q     258
    R     342
    S     339
    T     271
    U     293
    V     351
    W     280
    X     291
    Y     217
    Z     296


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
colors  = plt.cm.tab20(np.linspace(0, 1, len(class_names)))
bars    = ax.bar(class_names, counts, color=colors, edgecolor='black', lw=0.6)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontsize=7)

ax.axhline(counts.mean(), color='red', ls='--', alpha=0.7,
           label=f'Rata-rata: {counts.mean():.0f}')
ax.set_title('Distribusi Dataset BISINDO per Kelas Alfabet', fontsize=13, fontweight='bold')
ax.set_xlabel('Kelas (Huruf)')
ax.set_ylabel('Jumlah Sampel')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '00_distribusi_dataset.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Pra-pemrosesan Data (Data Preprocessing)
- **Train-Test Split**: 80% data latih / 20% data uji (Stratified, random_state=42)
- **StandardScaler**: Digunakan untuk KNN, Gaussian Naive Bayes, dan SVM (standarisasi mean=0, std=1)
- **MinMaxScaler**: Digunakan khusus Complement Naive Bayes (membutuhkan fitur non-negatif [0, 1])
- **Binarizer**: Digunakan khusus Bernoulli Naive Bayes (binarisasi fitur biner berbasis threshold=0.0)

In [ ]:
# Pembagian Data Latih dan Uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 1. StandardScaler (KNN, Gaussian NB, SVM)
sc      = StandardScaler()
X_tr    = sc.fit_transform(X_train)
X_te    = sc.transform(X_test)

# 2. MinMaxScaler (Complement NB)
mm      = MinMaxScaler()
X_tr_mm = mm.fit_transform(X_train)
X_te_mm = mm.transform(X_test)

# 3. StandardScaler + Binarizer (Bernoulli NB)
bn      = Binarizer(threshold=0.0)
X_tr_bn = bn.fit_transform(X_tr)
X_te_bn = bn.transform(X_te)

# Konfigurasi Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"[INFO] Data Latih : {X_tr.shape[0]} sampel")
print(f"[INFO] Data Uji   : {X_te.shape[0]} sampel")
print(f"[INFO] Dimensi    : {X_tr.shape[1]} fitur")

## 3. Konfigurasi 9 Model Klasifikasi
| No | Algoritma | Varian Model | Konfigurasi Parameter Utama |
|:--:|-----------|--------------|-----------------------------|
| 1  | KNN | k=3 | metric='euclidean', weights='distance' |
| 2  | KNN | k=5 | metric='euclidean', weights='distance' |
| 3  | KNN | k=7 | metric='euclidean', weights='distance' |
| 4  | Naive Bayes | Gaussian | var_smoothing=1e-9 |
| 5  | Naive Bayes | Complement | alpha=1.0 |
| 6  | Naive Bayes | Bernoulli | alpha=1.0, threshold=0.0 |
| 7  | SVM | Linear | kernel='linear', C=1.0 |
| 8  | SVM | RBF | kernel='rbf', C=10.0, gamma='scale' |
| 9  | SVM | Polynomial | kernel='poly', degree=3, C=1.0, gamma='scale' |

In [ ]:
MODEL_CONFIGS = [
    # K-Nearest Neighbors
    {
        'name'  : 'KNN (k=3)',
        'algo'  : 'KNN',
        'model' : KNeighborsClassifier(n_neighbors=3, metric='euclidean',
                                        weights='distance', algorithm='ball_tree', n_jobs=-1),
        'X_tr'  : X_tr, 'X_te': X_te,
        'cv_pipe': Pipeline([('sc', StandardScaler()),
                              ('m', KNeighborsClassifier(n_neighbors=3, metric='euclidean',
                                                          weights='distance', n_jobs=-1))])
    },
    {
        'name'  : 'KNN (k=5)',
        'algo'  : 'KNN',
        'model' : KNeighborsClassifier(n_neighbors=5, metric='euclidean',
                                        weights='distance', algorithm='ball_tree', n_jobs=-1),
        'X_tr'  : X_tr, 'X_te': X_te,
        'cv_pipe': Pipeline([('sc', StandardScaler()),
                              ('m', KNeighborsClassifier(n_neighbors=5, metric='euclidean',
                                                          weights='distance', n_jobs=-1))])
    },
    {
        'name'  : 'KNN (k=7)',
        'algo'  : 'KNN',
        'model' : KNeighborsClassifier(n_neighbors=7, metric='euclidean',
                                        weights='distance', algorithm='ball_tree', n_jobs=-1),
        'X_tr'  : X_tr, 'X_te': X_te,
        'cv_pipe': Pipeline([('sc', StandardScaler()),
                              ('m', KNeighborsClassifier(n_neighbors=7, metric='euclidean',
                                                          weights='distance', n_jobs=-1))])
    },
    # Naive Bayes
    {
        'name'  : 'Naive Bayes Gaussian',
        'algo'  : 'NaiveBayes',
        'model' : GaussianNB(var_smoothing=1e-9),
        'X_tr'  : X_tr, 'X_te': X_te,
        'cv_pipe': Pipeline([('sc', StandardScaler()), ('m', GaussianNB(var_smoothing=1e-9))])
    },
    {
        'name'  : 'Naive Bayes Complement',
        'algo'  : 'NaiveBayes',
        'model' : ComplementNB(alpha=1.0),
        'X_tr'  : X_tr_mm, 'X_te': X_te_mm,
        'cv_pipe': Pipeline([('mm', MinMaxScaler()), ('m', ComplementNB(alpha=1.0))])
    },
    {
        'name'  : 'Naive Bayes Bernoulli',
        'algo'  : 'NaiveBayes',
        'model' : BernoulliNB(alpha=1.0),
        'X_tr'  : X_tr_bn, 'X_te': X_te_bn,
        'cv_pipe': Pipeline([('sc', StandardScaler()), ('bn', Binarizer(threshold=0.0)),
                              ('m', BernoulliNB(alpha=1.0))])
    },
    # Support Vector Machine
    {
        'name'  : 'SVM Linear',
        'algo'  : 'SVM',
        'model' : SVC(kernel='linear', C=1.0, random_state=42),
        'X_tr'  : X_tr, 'X_te': X_te,
        'cv_pipe': Pipeline([('sc', StandardScaler()),
                              ('m', SVC(kernel='linear', C=1.0, random_state=42))])
    },
    {
        'name'  : 'SVM RBF',
        'algo'  : 'SVM',
        'model' : SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42),
        'X_tr'  : X_tr, 'X_te': X_te,
        'cv_pipe': Pipeline([('sc', StandardScaler()),
                              ('m', SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42))])
    },
    {
        'name'  : 'SVM Polynomial',
        'algo'  : 'SVM',
        'model' : SVC(kernel='poly', C=1.0, degree=3, gamma='scale', random_state=42),
        'X_tr'  : X_tr, 'X_te': X_te,
        'cv_pipe': Pipeline([('sc', StandardScaler()),
                              ('m', SVC(kernel='poly', C=1.0, degree=3,
                                        gamma='scale', random_state=42))])
    },
]

print(f"[INFO] Sebanyak {len(MODEL_CONFIGS)} model siap dilatih:")
for cfg in MODEL_CONFIGS:
    print(f"       - {cfg['name']}")

## 4. Pelatihan Model dan Evaluasi Awal

In [ ]:
results = []

print("Memulai proses pelatihan 9 model...\n")
print(f"{'Model':<28} | {'Accuracy':>8} | {'Precision':>9} | {'Recall':>8} | {'F1-Score':>8} | {'Waktu Latih':>11}")
print("-" * 85)

for cfg in MODEL_CONFIGS:
    # Pelatihan
    t0 = time.time()
    cfg['model'].fit(cfg['X_tr'], y_train)
    train_time = time.time() - t0

    # Prediksi
    t0 = time.time()
    y_pred = cfg['model'].predict(cfg['X_te'])
    pred_time = time.time() - t0

    # Kalkulasi Metrik
    acc   = accuracy_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec   = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1    = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    results.append({
        'name'       : cfg['name'],
        'algo'       : cfg['algo'],
        'accuracy'   : acc,
        'precision'  : prec,
        'recall'     : rec,
        'f1_score'   : f1,
        'train_time' : train_time,
        'pred_time'  : pred_time,
        'y_pred'     : y_pred
    })

    print(f"{cfg['name']:<28} | {acc:>8.4f} | {prec:>9.4f} | {rec:>8.4f} | {f1:>8.4f} | {train_time:>10.3f}s")

print("\n[INFO] Seluruh model telah berhasil dilatih.")

## 5. Validasi Silang (Stratified 5-Fold Cross-Validation)

In [ ]:
print("Menjalankan 5-Fold Stratified Cross-Validation...\n")
print(f"{'Model':<28} | {'CV F1 Mean':>12} | {'CV F1 Std':>11}")
print("-" * 58)

for cfg, res in zip(MODEL_CONFIGS, results):
    cv_scores = cross_val_score(
        cfg['cv_pipe'], X, y,
        cv=cv, scoring='f1_weighted', n_jobs=-1
    )
    res['cv_f1_mean'] = float(cv_scores.mean())
    res['cv_f1_std']  = float(cv_scores.std())
    res['cv_scores']  = cv_scores.tolist()

    print(f"{cfg['name']:<28} | {cv_scores.mean():>12.4f} | {cv_scores.std():>11.4f}")

print("\n[INFO] Validasi silang selesai.")

## 6. Tabel Perbandingan Kinerja Seluruh Model

In [ ]:
df_results = pd.DataFrame([
    {
        'Model'         : r['name'],
        'Algoritma'     : r['algo'],
        'Accuracy'      : r['accuracy'],
        'Precision'     : r['precision'],
        'Recall'        : r['recall'],
        'F1-Score'      : r['f1_score'],
        'CV F1 Mean'    : r['cv_f1_mean'],
        'CV F1 Std'     : r['cv_f1_std'],
        'Train Time (s)': round(r['train_time'], 3),
        'Pred Time (s)' : round(r['pred_time'], 3),
    }
    for r in results
])

df_display = df_results.sort_values('F1-Score', ascending=False).reset_index(drop=True)
df_display.index = df_display.index + 1

display(
    df_display.drop('Algoritma', axis=1)
    .style
    .format({
        'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}',
        'F1-Score': '{:.4f}', 'CV F1 Mean': '{:.4f}', 'CV F1 Std': '{:.4f}'
    })
    .highlight_max(
        subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'CV F1 Mean'],
        color='#e2f0d9'
    )
    .highlight_min(
        subset=['Train Time (s)', 'Pred Time (s)'],
        color='#deebf7'
    )
    .set_caption('Tabel 1: Peringkat Performa 9 Model Klasifikasi (Diurutkan berdasarkan F1-Score)')
    .set_properties(**{'text-align': 'center'})
)

df_display.to_csv(os.path.join(OUTPUT_DIR, 'comparison_table.csv'), index=True)
print("[INFO] Tabel perbandingan disimpan ke comparison_table.csv")

## 7. Visualisasi Hasil Evaluasi

In [ ]:
ALGO_COLORS = {'KNN': '#2b5c8f', 'NaiveBayes': '#6f42c1', 'SVM': '#d9534f'}
bar_colors  = [ALGO_COLORS[r['algo']] for r in results]
names       = [r['name'] for r in results]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for ax, metric, label in zip(
    axes,
    ['accuracy', 'precision', 'recall', 'f1_score'],
    ['Accuracy', 'Precision', 'Recall', 'F1-Score']
):
    vals = [r[metric] for r in results]
    bars = ax.bar(range(len(names)), vals, color=bar_colors,
                  edgecolor='black', lw=0.7, alpha=0.9)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_title(f'Perbandingan {label}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Nilai')
    ax.axhline(np.mean(vals), color='black', ls='--', alpha=0.5,
               label=f'Rata-rata: {np.mean(vals):.3f}')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

legend_patches = [mpatches.Patch(color=c, label=l)
                  for l, c in [('KNN', '#2b5c8f'), ('Naive Bayes', '#6f42c1'), ('SVM', '#d9534f')]]
fig.legend(handles=legend_patches, loc='upper center', ncol=3,
           fontsize=11, bbox_to_anchor=(0.5, 1.02))
plt.suptitle('Perbandingan 4 Metrik Evaluasi: KNN vs Naive Bayes vs SVM',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_comparison_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_heat = pd.DataFrame(results)[['name', 'accuracy', 'precision', 'recall', 'f1_score', 'cv_f1_mean']]
df_heat = df_heat.set_index('name')
df_heat.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'CV F1']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_heat, annot=True, fmt='.4f', cmap='Blues', ax=ax,
            linewidths=0.5, linecolor='gray',
            cbar_kws={'label': 'Nilai Metrik'},
            annot_kws={'size': 10, 'fontweight': 'bold'})
ax.set_title('Peta Panas (Heatmap) Metrik Evaluasi 9 Model', fontsize=13, fontweight='bold')
ax.set_ylabel('Model Klasifikasi')
plt.xticks(fontsize=10)
plt.yticks(fontsize=10, rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_heatmap_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'CV F1']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
cmap = plt.get_cmap('tab10')

for i, r in enumerate(results):
    vals = [r['accuracy'], r['precision'], r['recall'], r['f1_score'], r['cv_f1_mean']]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', lw=1.8, color=cmap(i % 10), label=r['name'])
    ax.fill(angles, vals, alpha=0.05, color=cmap(i % 10))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0.3, 1.0)
ax.set_title('Radar Chart — Profil Performa 9 Model Klasifikasi',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.55, 1.15), fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_radar_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
algos = ['KNN', 'NaiveBayes', 'SVM']
algo_labels = ['KNN', 'Naive Bayes', 'SVM']
metrics_plot = ['accuracy', 'precision', 'recall', 'f1_score']
metric_labels_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors4 = ['#2b5c8f', '#28a745', '#fd7e14', '#dc3545']

# Model terbaik per algoritma berdasarkan F1-Score
best_vals = {
    algo: {m: max(r[m] for r in results if r['algo'] == algo) for m in metrics_plot}
    for algo in algos
}
best_names = {
    algo: max([r for r in results if r['algo'] == algo], key=lambda r: r['f1_score'])['name']
    for algo in algos
}

x = np.arange(len(algo_labels))
width = 0.18
offsets = [-0.27, -0.09, 0.09, 0.27]

fig, ax = plt.subplots(figsize=(12, 6))
for i, (m, ml, c) in enumerate(zip(metrics_plot, metric_labels_plot, colors4)):
    vals = [best_vals[a][m] for a in algos]
    bars = ax.bar(x + offsets[i], vals, width, label=ml, color=c,
                  edgecolor='black', lw=0.7, alpha=0.9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([f'{l}\n(Model Terbaik: {best_names[a]})'
                     for l, a in zip(algo_labels, algos)], fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Nilai Metrik', fontsize=11)
ax.set_title('Perbandingan Model Terbaik per Algoritma', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_grouped_best_per_algo.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (xkey, xlabel) in zip(axes, [('train_time', 'Waktu Pelatihan (detik)'),
                                       ('pred_time',  'Waktu Inferensi/Prediksi (detik)')]):
    for r in results:
        ax.scatter(r[xkey], r['f1_score'], s=120,
                   color=ALGO_COLORS[r['algo']], alpha=0.85,
                   edgecolors='black', lw=0.7, zorder=5)
        ax.annotate(r['name'],
                    xy=(r[xkey], r['f1_score']),
                    xytext=(6, 4), textcoords='offset points', fontsize=8)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('F1-Score (Weighted)', fontsize=11)
    ax.set_title(f'Trade-off: {xlabel} vs F1-Score', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

legend_patches = [mpatches.Patch(color=c, label=l)
                  for l, c in [('KNN', '#2b5c8f'), ('Naive Bayes', '#6f42c1'), ('SVM', '#d9534f')]]
axes[0].legend(handles=legend_patches, fontsize=10)
plt.suptitle('Analisis Trade-off: Waktu Komputasi vs Performa Klasifikasi (F1-Score)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_tradeoff_time_f1.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Laporan Klasifikasi per Kelas Huruf (A–Z)

In [ ]:
for r in results:
    print(f"\n{'='*70}")
    print(f"  Klasifikasi Laporan: {r['name']}")
    print(f"{'='*70}")
    print(classification_report(y_test, r['y_pred'], target_names=class_names))

    report = classification_report(y_test, r['y_pred'],
                                   target_names=class_names, output_dict=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, metric, color in zip(axes,
                                  ['precision', 'recall', 'f1-score'],
                                  ['#2b5c8f',   '#28a745', '#fd7e14']):
        vals = [report[cls][metric] for cls in class_names]
        ax.bar(class_names, vals, color=color, alpha=0.85, edgecolor='black', lw=0.5)
        ax.axhline(np.mean(vals), color='black', ls='--', alpha=0.6,
                   label=f'Rata-rata: {np.mean(vals):.3f}')
        ax.set_ylim(0, 1.1)
        ax.set_title(f'{metric.replace("-", " ").title()} per Kelas', fontweight='bold', fontsize=11)
        ax.legend(fontsize=8)
        ax.tick_params(axis='x', labelsize=7)

    plt.suptitle(f'Metrik per Kelas — {r["name"]}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fname = r['name'].lower().replace(' ', '_').replace('(', '').replace(')', '').replace('=', '')
    plt.savefig(os.path.join(OUTPUT_DIR, f'06_per_class_{fname}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 9. Matriks Konfusi (Confusion Matrix)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(24, 22))
axes = axes.flatten()

cmaps_map = {'KNN': 'Blues', 'NaiveBayes': 'Purples', 'SVM': 'Reds'}

for ax, r in zip(axes, results):
    cm     = confusion_matrix(y_test, r['y_pred'])
    cmap   = cmaps_map[r['algo']]
    im     = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    thresh = cm.max() / 2.0

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center', fontsize=5,
                    color='white' if cm[i, j] > thresh else 'black')

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(class_names, fontsize=6)
    ax.set_xlabel('Prediksi', fontsize=8)
    ax.set_ylabel('Label Sebenarnya', fontsize=8)
    ax.set_title(
        f"{r['name']}\nAccuracy={r['accuracy']:.4f} | F1-Score={r['f1_score']:.4f}",
        fontsize=9, fontweight='bold'
    )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Matriks Konfusi Seluruh 9 Model (BISINDO 26 Kelas)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_confusion_matrices_all.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_per_algo = {
    algo: max([r for r in results if r['algo'] == algo], key=lambda r: r['f1_score'])
    for algo in ['KNN', 'NaiveBayes', 'SVM']
}

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, (algo, r) in zip(axes, best_per_algo.items()):
    cm     = confusion_matrix(y_test, r['y_pred'])
    im     = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    thresh = cm.max() / 2.0

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center', fontsize=6,
                    color='white' if cm[i, j] > thresh else 'black')

    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel('Prediksi', fontsize=9)
    ax.set_ylabel('Label Sebenarnya', fontsize=9)
    ax.set_title(
        f"Model Terbaik {algo}: {r['name']}\nAccuracy={r['accuracy']:.4f} | F1={r['f1_score']:.4f}",
        fontsize=10, fontweight='bold'
    )
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Matriks Konfusi — Model Terbaik per Algoritma', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08_cm_best_per_algo.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Kesimpulan dan Rekomendasi

In [ ]:
best_model = max(results, key=lambda r: r['f1_score'])
best_cv    = max(results, key=lambda r: r['cv_f1_mean'])
fastest    = min(results, key=lambda r: r['train_time'])

print("=" * 75)
print("  KESIMPULAN: EVALUASI KOMPARATIF 9 MODEL KLASIFIKASI BISINDO")
print("=" * 75)
print(f"  Performa F1-Score Tertinggi   : {best_model['name']:<25} ({best_model['f1_score']:.4f})")
print(f"  Konsistensi Validasi Silang   : {best_cv['name']:<25} (CV F1: {best_cv['cv_f1_mean']:.4f})")
print(f"  Waktu Komputasi Tercepat      : {fastest['name']:<25} ({fastest['train_time']:.3f} detik)")
print("=" * 75)

print("\nModel Terbaik pada Setiap Algoritma:")
print(f"  {'Algoritma':<15} | {'Model':<25} | {'Accuracy':>8} | {'F1-Score':>8} | {'CV F1':>8}")
print(f"  {'-'*72}")
for algo in ['KNN', 'NaiveBayes', 'SVM']:
    b = max([r for r in results if r['algo'] == algo], key=lambda r: r['f1_score'])
    print(f"  {algo:<15} | {b['name']:<25} | {b['accuracy']:>8.4f} | {b['f1_score']:>8.4f} | {b['cv_f1_mean']:>8.4f}")

print("\nPeringkat Lengkap Seluruh Model:")
df_final = pd.DataFrame(results).sort_values('f1_score', ascending=False)
display(df_final[['name', 'accuracy', 'precision', 'recall', 'f1_score', 'cv_f1_mean', 'train_time']]
        .rename(columns={'name': 'Model', 'accuracy': 'Accuracy', 'precision': 'Precision',
                          'recall': 'Recall', 'f1_score': 'F1-Score',
                          'cv_f1_mean': 'CV F1 Mean', 'train_time': 'Train Time (s)'})
        .reset_index(drop=True)
        .style.format({'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}',
                        'F1-Score': '{:.4f}', 'CV F1 Mean': '{:.4f}', 'Train Time (s)': '{:.3f}'})
        .highlight_max(subset=['Accuracy', 'F1-Score', 'CV F1 Mean'], color='#e2f0d9')
        .set_caption('Tabel 2: Peringkat Akhir Kinerja Model Klasifikasi BISINDO'))

# Ekspor hasil ke file JSON
export = [{k: v for k, v in r.items() if k != 'y_pred'} for r in results]
with open(os.path.join(OUTPUT_DIR, 'final_results.json'), 'w') as f:
    json.dump(export, f, indent=2)
print(f"\n[INFO] Seluruh proses selesai. Hasil komparatif disimpan di: {OUTPUT_DIR}")